# Visão geral dos resultados agregados

Este caderno usa só os arquivos de `resultados_agregados/`, que não têm nenhum dado por participante. Ele não precisa dos bancos de EEG nem do pacote pré-processado: serve para conferir os números do artigo e para entender como as duas figuras centrais são construídas.

O que ele faz:

1. mostra o tamanho das amostras de cada banco depois do pipeline;
2. refaz o **tabuleiro de variáveis** (quanto cada característica depende da amostra, do artefato e do processamento);
3. refaz o **forest plot das duas metades de um confundidor** (o fator difere entre os grupos? o efeito muda quando ele é ajustado?).

As versões finais das figuras, no formato do artigo, saem de `scripts/03_figuras.py`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RES = RAIZ / 'resultados_agregados'
plt.rcParams.update({'font.family': 'serif', 'font.size': 8.5, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Amostras

Contagens de TDAH e controles que entram nas análises, por banco e condição (OF: olhos fechados; OA: olhos abertos). No TDBRAIN entram só adultos (18 anos ou mais); o adhdata é o único banco em tarefa e não informa idade nem sexo por participante.

In [ ]:
pd.DataFrame(json.loads((RES / 'e2_amostras.json').read_text(encoding='utf-8'))).T

## 2. O tabuleiro de variáveis

Cada linha é uma variável e cada coluna é uma característica medida em um banco (olhos fechados no HBN e no TDBRAIN, tarefa no adhdata). O valor de cada casa depende do tipo de linha:

- **amostra e artefato** (idade, sexo, duração, músculo, olho, potência total): correlação de Spearman (ρ) com a característica, calculada só nos controles, para que o diagnóstico não entre na conta; verde |ρ| < 0,2, amarelo 0,2 a 0,4, vermelho 0,4 ou mais;
- **processamento** (rejeitar épocas com músculo ou olho, ICA ocular, só canais internos, só 60 s, referência nativa, ajuste de 1 a 30 Hz): mudança no g de Hedges ajustado (Δg) quando a versão base é trocada pela variante; verde |Δg| < 0,10, amarelo 0,10 a 0,20, vermelho 0,20 ou mais. Na família espacial a mudança é na AUC (limiares 0,03 e 0,07);
- **sinal** (azul): o próprio efeito de grupo na base, g ajustado por idade, idade², sexo, idade × sexo e duração; na família espacial, a AUC.

Casas em branco são combinações não disponíveis, por exemplo idade e sexo no adhdata.

In [ ]:
longo = pd.read_csv(RES / 'e6_tabuleiro_longo.csv')
blocos = {'adhdata_tarefa': 'a', 'hbn_fechado': 'H', 'tdbrain_fechado': 'T'}
feats = ['chi', 'chi_post', 'offset', 'alfa_pw', 'iaf', 'tbr', 'espacial']
linhas = ['idade', 'sexo_f', 'log_dur', 'emg_t', 'ocular', 'log_var', 'rej_emg', 'rej_olho', 'ica_olho', 'internos',
          'dur60', 'nativa', 'U30', 'sinal']
limites = {'rho_controles': (0.2, 0.4), 'delta_g': (0.10, 0.20), 'delta_auc': (0.03, 0.07)}


def classe(tipo, v):
    if tipo not in limites:
        return 3
    lo, hi = limites[tipo]
    return 0 if abs(v) < lo else 1 if abs(v) < hi else 2


colunas = [(f, b) for f in feats for b in blocos]
M = np.full((len(linhas), len(colunas)), np.nan)
V = np.full_like(M, np.nan)
for i, lin in enumerate(linhas):
    for j, (f, b) in enumerate(colunas):
        s = longo[(longo['bloco'] == b) & (longo['feature'] == f) & (longo['coluna'] == lin)]
        if len(s) and np.isfinite(s['valor'].iloc[0]):
            V[i, j] = s['valor'].iloc[0]
            M[i, j] = classe(s['tipo'].iloc[0], V[i, j])

fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(np.ma.masked_invalid(M), cmap=ListedColormap(['#4DAF7C', '#F2C14E', '#D1495B', '#BFD3E6']), vmin=0, vmax=3,
          aspect='auto')
for i, j in zip(*np.nonzero(np.isfinite(V))):
    ax.text(j, i, f'{V[i, j]:.2f}'.replace('.', ','), ha='center', va='center', fontsize=6.5)
ax.set_xticks(range(len(colunas)))
ax.set_xticklabels([f + chr(10) + blocos[b] for f, b in colunas], fontsize=7)
ax.set_yticks(range(len(linhas)))
ax.set_yticklabels(linhas)
for x in np.arange(2.5, len(colunas), 3):
    ax.axvline(x, color='white', lw=2)
for y in (2.5, 5.5, 12.5):
    ax.axhline(y, color='black', lw=0.8)
ax.set_title('Tabuleiro de variáveis (a = adhdata, tarefa; H = HBN e T = TDBRAIN, olhos fechados)', loc='left')
plt.show()

Como ler: a idade (primeira linha) acompanha as características espectrais nos dois bancos de repouso, mas não a família espacial; no bloco de processamento, as limpezas (rejeitar épocas, ICA ocular, só 60 s) quase não mudam o efeito, enquanto a referência nativa muda bastante e a faixa de ajuste de 1 a 30 Hz muda o expoente do adhdata. A última linha é o sinal diagnóstico na base.

## 3. As duas metades de um confundidor

Um fator só confunde o efeito de grupo se as duas coisas acontecerem ao mesmo tempo: ele acompanha a característica (o tabuleiro mostra isso) **e** difere entre TDAH e controles.

- O painel A mostra a segunda metade: a diferença padronizada, sem ajuste, de cada fator entre os grupos (faixa cinza: |d| < 0,1).
- O painel B mostra a consequência: o g de Hedges do expoente, do offset e da potência do alfa com três conjuntos de covariáveis: C0 (idade, idade², sexo, idade × sexo e duração), C0 com a potência total, e C0 com a potência total e os índices de músculo e olho. Barra: IC de 95 % por bootstrap; ponto cheio: o IC exclui zero.

In [ ]:
des = pd.read_csv(RES / 'e10_desequilibrio.csv')
aj = pd.read_csv(RES / 'e9_ajuste_potencia.csv')
blocos5 = ['adhdata_tarefa', 'hbn_fechado', 'hbn_aberto', 'tdbrain_fechado', 'tdbrain_aberto']
variaveis = ['idade', 'sexo_f', 'log_dur', 'emg_t', 'ocular', 'log_var']
ajustes = [('C0', '#5B6770'), ('C0+pot', '#2F7FB8'), ('C0+pot+art', '#C8553D')]
cores = ['#7A7A7A', '#1F4E79', '#6FA3D2', '#8C2D19', '#D98C73']

fig = plt.figure(figsize=(10, 7.5))
grade = fig.add_gridspec(2, 3, height_ratios=[1, 1.3], hspace=0.5, wspace=0.1)
ax = fig.add_subplot(grade[0, :])
for i, b in enumerate(blocos5):
    s = des[des['bloco'] == b].set_index('variavel').reindex(variaveis)
    ax.bar(np.arange(len(variaveis)) - 0.4 + 0.16 * (i + 0.5), s['smd'], width=0.15, color=cores[i], label=b)
ax.axhspan(-0.1, 0.1, color='#EEEEEE', zorder=0)
ax.axhline(0, color='#444444', lw=0.6)
ax.set_xticks(range(len(variaveis)))
ax.set_xticklabels(variaveis)
ax.set_ylabel('diferença padronizada')
ax.legend(ncol=5, fontsize=7, frameon=False)
ax.set_title('A  O fator difere entre os grupos?', loc='left')
for k, f in enumerate(['chi', 'offset', 'alfa_pw']):
    ax = fig.add_subplot(grade[1, k])
    for i, b in enumerate(blocos5):
        for m, (a, cor) in enumerate(ajustes):
            s = aj[(aj['bloco'] == b) & (aj['feature'] == f) & (aj['ajuste'] == a)]
            if len(s):
                g, lo, hi = s[['g', 'g_lo', 'g_hi']].iloc[0]
                y = i + (m - 1) * 0.22
                ax.plot([lo, hi], [y, y], color=cor, lw=1.1)
                ax.plot(g, y, 'o', color=cor, ms=4, mfc=cor if (lo > 0 or hi < 0) else 'white')
    ax.axvline(0, color='#444444', lw=0.6)
    ax.set_yticks(range(len(blocos5)))
    ax.set_yticklabels(blocos5 if k == 0 else [])
    ax.invert_yaxis()
    ax.set_xlabel('g de Hedges (TDAH menos controle)')
    ax.set_title(('B  O efeito muda quando o fator é ajustado?' + chr(10) if k == 0 else chr(10)) + f, loc='left', fontsize=9)
fig.legend([plt.Line2D([], [], color=c, marker='o') for _, c in ajustes], [a for a, _ in ajustes], ncol=3, frameon=False,
           loc='lower center')
plt.show()

Como ler: no TDBRAIN, a potência total é menor no grupo TDAH (painel A) e o efeito do offset muda quando ela entra nas covariáveis (painel B); no HBN, os fatores de aquisição quase não diferem entre os grupos e o ajuste muda pouco. No adhdata, que é de tarefa, o movimento pode fazer parte do próprio quadro: ajustar por ele pode remover efeito verdadeiro, e por isso o ajuste não é automaticamente a resposta certa.

## 4. Escore de cuidado (Tabela 3 do artigo)

O escore resume o tabuleiro numa nota de 0 a 1 por característica. Cada casa amarela vale 1 e cada vermelha vale 2; a soma é dividida pelo máximo possível (2 vezes o número de casas disponíveis). As colunas Amostra, Artefato e aquisição e Processamento usam o tabuleiro de olhos fechados nos três bancos (adhdata em tarefa); Total OA usa o HBN e o TDBRAIN de olhos abertos. Quanto maior o escore, mais a característica depende de fatores alheios ao diagnóstico e mais cuidado ela pede.

In [ ]:
esc = pd.read_csv(RES / 'e20_escore_cuidado.csv')
nomes = {'offset': 'Offset', 'chi': 'Expoente χ', 'espacial': 'Distância espacial', 'chi_post': 'Expoente posterior',
         'tbr': 'TBR', 'alfa_pw': 'Potência do alfa', 'iaf': 'IAF'}
of = esc[esc['estado'] == 'OF'].pivot(index='feature', columns='grupo_linhas', values='escore')
tabela3 = pd.DataFrame({'Amostra': of['amostra'], 'Artefato e aquisição': of['artefato_aquisicao'],
                        'Processamento': of['processamento'], 'Total OF': of['total'],
                        'Total OA': esc[esc['estado'] == 'OA'].set_index('feature').query("grupo_linhas == 'total'")['escore']})
tabela3.rename(index=nomes).sort_values('Total OF', ascending=False).round(2)